> 本 Notebook 由对应 Word 实验手册生成。只有“测试与验收”章节中的测试指令可执行；其余代码仅用于阅读和讲解。


# 实验十：基于CANN的GQA-Attention优化版算子实验


建议学时：4学时


# 实验任务


## 任务描述


本实验在GQA Attention基础版基础上进行优化：通过profiling定位基础版的标量GM访问瓶颈，将逐元素GetValue/SetValue的QK点积和V累积替换为基于UB缓冲区和Vector API的高效实现，并保持在线softmax的数值稳定性和causal mask语义不变。


## 学习目标


完成本任务的学习后，你应当能使用msprof采集基础版Attention kernel的AI Core指标，定位主要瓶颈、理解UB缓冲区层次与TPipe/TQue/TBuf异步流水线在算子优化中的作用、实现DataCopy批量搬运Q/K/V行数据从GM到UB，替代逐元素GetValue、实现向量化QK点积（Mul + ReduceSum替代headDim次标量乘加）、实现向量化在线softmax与V累积（Muls + Add替代逐元素标量更新）、对比优化前后standalone benchmark数据，计算加速比并解释瓶颈迁移、建立profiling驱动、逐项迭代、可量化验收的算子优化流程。


# 任务准备


## 优化前的瓶颈与策略总览


优化动机：基础版为了便于验证在线softmax的数值稳定公式，对每个 (batch, qHead, qPos) 三元组，在每个key位置上用headDim次标量GetValue读取Q和K、标量循环累加QK点积、再用headDim次标量I/O完成V累积的原地更新。对于标准形状B=1, Hq=8, Sq=Sk=32, D=64，每个query对每个key执行64次Q读 + 64次K读 + softmax更新 + 64次V读 + 64次output写，GM访问量为O(Sq×Sk×headDim) ≈ 32×32×64 = 65,536次标量访问。msprof采集显示aiv_scalar_time占比接近100%，向量单元完全空闲，内存有效带宽远低于HBM标称值。优化版将逐元素的标量GM访问替换为"DMA批量搬入UB → 向量API并行计算 → DMA批量写回"。


## 基础版与优化版的前后对比


优化版不改变公式、GQA head映射和causal mask语义，而是将基础版的逐元素标量路径替换为UB缓冲 + 向量化三阶段流水线。下表给出每项策略对应的基础版位置、优化版改动和预期作用。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">优化策略</th>
<th style="text-align:left;">基础版的实现</th>
<th style="text-align:left;">优化版的修改位置</th>
<th style="text-align:left;">改动带来的作用</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">UB缓冲区与TPipe流水线</td>
<td style="text-align:left;">直接通过标量IO接口，对Q、K、V及输出张量进行逐元素读写</td>
<td style="text-align:left;">在初始化阶段，利用TPipe申请各部分双缓冲区 。处理流程重构为经典三段式流水线</td>
<td style="text-align:left;">数据的所有权及其生命周期交由队列统一管理，为后续引入双缓冲机制预留了接口。</td>
</tr>
<tr>
<td style="text-align:left;">QK点积的向量化</td>
<td style="text-align:left;">对每个key位置，采用 headDim 次标量乘加循环计算分值。</td>
<td style="text-align:left;">先通过DMA将K行搬入UB，再调用向量API一次性完成 headDim 个元素的逐分量乘法，以及向量化归约求和到 sum[0]。</td>
<td style="text-align:left;">将计算复杂度降为 O(Sq × Sk) 次DMA加向量计算。</td>
</tr>
<tr>
<td style="text-align:left;">在线Softmax与V累积的向量化</td>
<td style="text-align:left;">采用标量方式逐元素更新输出。</td>
<td style="text-align:left;">使用 Muls对累积器中的 headDim 个元素同时乘以旧因子；再对当前V行进行缩放。</td>
<td style="text-align:left;">消除了逐元素标量路径中 O(Sq × Sk × headDim) 次对输出张量的读/写操作。</td>
</tr>
</tbody></table>


## 算子定义与接口约定


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">项目</th>
<th style="text-align:left;">当前工程约定</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">数据类型</td>
<td style="text-align:left;">Q、K、V、output 均为 float32</td>
</tr>
<tr>
<td style="text-align:left;">输入布局</td>
<td style="text-align:left;">四维连续 Tensor：与基础版完全一致</td>
</tr>
<tr>
<td style="text-align:left;">行数</td>
<td style="text-align:left;">totalQueries = B × Hq × Sq</td>
</tr>
<tr>
<td style="text-align:left;">并行划分</td>
<td style="text-align:left;">coreNum = min(8, totalQueries)；queriesPerCore = (totalQueries + coreNum - 1) / coreNum 向上取整；每个 Core 从 coreId 推导起始的 (batch, qHead)，末核自动截断</td>
</tr>
<tr>
<td style="text-align:left;">附加约束</td>
<td style="text-align:left;">headDim 必须被 8 整除（DataCopy 对齐要求）；UB 缓冲区每个 slot 大小为 headDim × sizeof(float)，须不超过 UB 容量（~256KB）</td>
</tr>
<tr>
<td style="text-align:left;">标准测试形状</td>
<td style="text-align:left;">B=1, Hq=8, Hkv=2, Sq=Sk=32, D=64, causal=1, coreNum=8</td>
</tr>
</tbody></table>


## 实验环境准备


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">项目</th>
<th style="text-align:left;">配置</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">硬件</td>
<td style="text-align:left;">Ascend 910B4宿主NPU</td>
</tr>
<tr>
<td style="text-align:left;">工具链</td>
<td style="text-align:left;">CANN 8.5.0 需先source set_env.sh</td>
</tr>
<tr>
<td style="text-align:left;">框架接口</td>
<td style="text-align:left;">PyTorch C++ extension，torch.ops.gqa_attention_optimized_custom.gqa_attention(Tensor q, Tensor k, Tensor v, float scale=0., bool causal=True) -&gt;Tensor；</td>
</tr>
<tr>
<td style="text-align:left;">构建结果</td>
<td style="text-align:left;">out/lib/libascendc_kernels_npu.so、libgqa_attention_torch_register.so；out/bin/gqa_attention_optimized_standalone</td>
</tr>
<tr>
<td style="text-align:left;">测试参考</td>
<td style="text-align:left;">PyTorch 标准Attention；新增对比脚本compare_with_baseline.sh同口径对比基础版与优化版device时间</td>
</tr>
</tbody></table>


# 任务实施


## 步骤一：Profiling采集与瓶颈定位


在基础版工程目录下运行测试可执行文件和msprof，采集设备端性能指标。基础版Query向量对Key向量执行头维度次标量读取Q/K，再执行头维度次GetValue/SetValue更新output。


关键参数：aiv_scalar_time占比接近100%，向量单元完全空闲，Memory有效带宽远低于HBM标称值，每query的GM访问次数为O(Sk×headDim)，呈线性增长。瓶颈明确：标量GM逐元素访问，改用UB缓冲区+ DataCopy批量搬运+Vector API计算。


## 步骤二：UB缓冲区与TPipe流水线


优化版引入TPipe/TQue/TBuf管理UB内存层次（同时也是CANN的三段式流水线下的标准管理方式）。Init阶段分配以下UB资源：


queryQue：TQue<VECIN,1>，存放当前query行


keyQue/valueQue：TQue<VECIN,1>，每次循环复用


productBuf/accBuf/tempBuf：TBuf<VECCALC>，作为中间计算缓存


sumBuf：ReduceSum输出


outputQue：TQue<VECOUT,1>，保存计算结果


每个TQue大小为headDim × sizeof(float)，一个slot即一行的全部元素。Process方法重构为CopyInQuery、Compute、CopyOut三阶段：


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">// CopyIn LocalTensor<float> query = queryQue_.AllocTensor<float>(); DataCopy(query, queryGm_[qBase], headDim_); queryQue_.EnQue(query); // Compute (inner loop with K/V) LocalTensor<float> queryIn = queryQue_.DeQue<float>(); // ... QK dot + softmax + V accum ... // CopyOut LocalTensor<float> out = outputQue_.AllocTensor<float>(); // ... finalize output ... DataCopy(outputGm_[index * headDim_], out, headDim_);</th>
</tr>
</thead>
</table>


三阶段结构保证了GM和UB之间的搬移与UB内计算的清晰分离：CopyIn用DataCopy将一行Q从GM批量搬入UB，Compute在UB内完成全部计算，CopyOut用DataCopy将结果写回GM。


## 步骤三：向量化QK点积


基础版中，每个Key向量位置的QK点积需要headDim次标量乘加循环。优化版使用Vector API的Mul + ReduceSum替代：


DataCopy将K行搬入UB后，Mul(product, query, key, headDim)一次性完成headDim次元素乘


ReduceSum(sum, product, reduce, headDim) 将product的headDim个元素归约求和到sum[0]。


两步向量操作替代了headDim次标量循环，包含在线softmax的max追踪和normalizer更新：


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">// Mul: element-wise Q*K vector operation Mul(product, query, key, static_cast<int32_t>(headDim_)); // ReduceSum: sum product elements → single score ReduceSum(sum, product, reduce, static_cast<int32_t>(headDim_)); float score = sum.GetValue(0) * scale_; float nextMax = score &gt; maxScore ? score : maxScore; float oldFactor = GqaOptimizedExp(maxScore - nextMax); float newFactor = GqaOptimizedExp(score - nextMax); normalizer = normalizer * oldFactor + newFactor;</th>
</tr>
</thead>
</table>


## 步骤四：向量化Softmax与V累积


在线softmax更新使用Muls + Add向量操作替代逐元素标量,将当前累积器acc的headDim个元素同时乘以oldFactor，Muls用newFactor缩放当前V行，Add将缩放后的V累加到acc。循环结束后Muls一次性完成归一化。流水线中的计算阶段的核心代码为：


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">// Vector softmax: Muls + Add Muls(acc, acc, oldFactor, static_cast<int32_t>(headDim_)); Muls(temp, value, newFactor, static_cast<int32_t>(headDim_)); Add(acc, acc, temp, static_cast<int32_t>(headDim_)); maxScore = nextMax; // ... loop end ... // Final normalize LocalTensor<float> out = outputQue_.AllocTensor<float>(); Muls(out, acc, 1.0f / normalizer, static_cast<int32_t>(headDim_));</th>
</tr>
</thead>
</table>


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">TORCH_LIBRARY(gqa_attention_optimized_custom, m) { m.def(&quot;gqa_attention(Tensor q, Tensor k, Tensor v, &quot; &quot;float scale=0., bool causal=True) -&gt; Tensor&quot;); } TORCH_LIBRARY_IMPL(gqa_attention_optimized_custom, CompositeExplicitAutograd, m) { m.impl(&quot;gqa_attention&quot;, gqa_attention_optimized_npu); }</th>
</tr>
</thead>
</table>


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">cd /YOURPATH/GqaAttentionOptimizedExperiment bash scripts/check_env.sh bash scripts/build.sh bash scripts/run_test.sh tests/test_torch_op.py # 单算子正确性 bash scripts/run_test.sh tests/test_qwen_forward.py # Qwen链路替换验证 bash scripts/compare_qwen2_5_forward.sh # Qwen2.5前向vsnative对比 bash scripts/compare_with_baseline.sh # 基础版vs优化版 性能对比 BATCH=1 Q_HEADS=8 KV_HEADS=2 Q_LEN=32 KV_LEN=32 HEAD_DIM=64 BLOCK_DIM=8 WARMUP=10 REPEAT=50 ROUNDS=5 CAUSAL=1 bash scripts/profile.sh</th>
</tr>
</thead>
</table>


# 测试与验收


## 一、单算子正确性测试


该测试调用当前实验的PyTorch注册算子，并与同一数学语义的参考实现比较。终端输出全部为PASS（或ALL PASS）且进程返回码为0，表示正确性测试通过。


In [ ]:
%%bash
set -e

cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops
source ./setup_cannlab_env.sh
cd Qwen2.5cann_ops/GqaAttentionOptimizedExperiment
bash scripts/build.sh
python3 tests/test_torch_op.py


## 二、单算子执行时间测试


该指令先预热，再重复启动单个算子，并使用ACL Event统计设备侧执行时间。记录输出中的mean、median、min和max；该结果不包含Python参考计算、输入生成、结果比对及首次主机到设备的数据传输。基础版与优化版比较时，应使用相同输入形状、预热次数、重复次数和计算核心数。


In [ ]:
%%bash
set -e

cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops
source ./setup_cannlab_env.sh
cd Qwen2.5cann_ops/GqaAttentionOptimizedExperiment
export LD_LIBRARY_PATH="$PWD/out/lib:${LD_LIBRARY_PATH:-}"
./out/bin/gqa_attention_optimized_standalone --batch 1 --q-heads 8 --kv-heads 2 --q-len 32 --kv-len 32 --head-dim 64 --block-dim 8 --causal 1 --warmup 10 --repeat 50 --rounds 5


# 任务拓展


UB流水线性能剖析：在优化版上运行msprof，对比基础版的aiv_scalar_time与优化版的aiv_vec_time / aic_mte_time占比变化。验证DataCopy批量搬运是否提升了Memory有效带宽，确认Vector Mul/ReduceSum/Muls/Add对AI Core向量单元的利用率。


Tiling规模扫描：改变batch、seq_len、headDim和GQA比例，运行优化版standalone benchmark。观察不同规模下UB缓冲区是否因headDim × sizeof(float) 超过UB容量而退化，评估per-query模型下的线性扩展能力。


进一步优化方向： Cube/MatMul替代Mul+ReduceSum（利用AI Core Cube单元加速QK^T矩阵乘时，应当注意NPU内部为分离式还是耦合式，来决定流水线分配）。


# 实验总结


本实验在GQA Attention基础版的基础上，完成了profiling驱动的向量化优化闭环：从msprof数据出发定位标量GM逐元素访问为主要瓶颈，将QK点积的headDim次标量乘加替换为单次Mul + ReduceSum向量操作，将V累积和softmax归一化的逐元素更新替换为Muls + Add向量操作，通过TPipe/TQue/TBuf管理UB缓冲区实现CopyIn、Compute→、CopyOut三阶段流水线。所有优化保持在线softmax的数值稳定性和causal mask语义不变，同口径standalone device时间从684.1us降至212.2us，加速3.224×。


优化版的核心价值在于展示了AscendC UB编程的基本模式：GM与UB之间通过DataCopy批量搬移，UB内通过Vector API并行计算，消弭了标量路径中O(Sk×headDim) 的GM访问放大效应。学生通过对比基础版和优化版的kernel代码，可以直观理解同一个在线softmax公式在标量GM和向量UB两种实现路径下的结构差异。
